# Deep Learning Challenge

1.   Listeneintrag
2.   Listeneintrag


DLMDSDL01 Deep Learning - Prof. Dr. Heinke Hihn
IU International University of Applied Sciences


# Objective of the Challenge

In this challenge, you will design, train, and systematically analyze a convolutional neural network (CNN) for multi-class image classification using the CIFAR-100 dataset (100 object classes).

The goal is not only to achieve high predictive performance, but to understand how architectural and training decisions influence:

convergence behavior

generalization performance

overfitting

training stability

You are expected to experiment systematically and document your findings in a structured manner.

# Your Tasks

You are provided with reusable code blocks for:

Dataset loading and deterministic splitting

A modular CNN architecture

Training and evaluation

Early stopping

One-epoch training experiments

In [ ]:
# 1. Setup

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Reproducibility

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [ ]:
# 2. Dataset Download

DATA_ROOT = "./data"

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5071, 0.4867, 0.4408],
        std=[0.2675, 0.2565, 0.2761]
    )
])

full_train_dataset = torchvision.datasets.CIFAR100(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR100(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=transform
)

print("Training samples:", len(full_train_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
# Deterministic train / val split

indices = list(range(len(full_train_dataset)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=full_train_dataset.targets
)

train_dataset = Subset(full_train_dataset, train_idx)
val_dataset   = Subset(full_train_dataset, val_idx)

BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
# Conv Block
# feel free to modify

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels,
                 activation="relu",
                 dropout=0.0):
        super().__init__()

        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(dropout)

        self.activation_name = activation
        self.activation = self.get_activation(activation)

    def get_activation(self, name):
        if name == "relu":
            return nn.ReLU()
        elif name == "leaky_relu":
            return nn.LeakyReLU()
        elif name == "gelu":
            return nn.GELU()
        else:
            raise ValueError("Unknown activation")

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.activation(x)
        x = self.dropout(x)
        return x

In [ ]:
# this is the base model. feell free to adapt as needed
class CNNModel(nn.Module):
    def __init__(self, num_classes,
                 activation="relu",
                 dropout=0.3):
        super().__init__()

        # the feature extractor
        self.features = nn.Sequential(
            ConvBlock(3, 64, activation, dropout),
            nn.MaxPool2d(2),
            ConvBlock(64, 128, activation, dropout),
            nn.MaxPool2d(2),
            ConvBlock(128, 256, activation, dropout),
            nn.MaxPool2d(2)
        )

        # the final layer
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),  # adjusted for 32x32 input
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

        self.apply(self.initialize_weights)

    def initialize_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# Training and Evaluation

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / len(loader), correct / total

In [ ]:
# Early Stopping

class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = np.inf
        self.counter = 0

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
            return False

In [ ]:
# Create and Train Model (with Early Stopping)

# 1. Model instantiation
num_classes = 100  # CIFAR-100

model = CNNModel(
    num_classes=num_classes,
    activation="relu",   # students can modify
    dropout=0.3          # students can modify
).to(device)

# 2. Loss function
criterion = nn.CrossEntropyLoss()

# 3. Optimizer
# You can change this to a different optimizer
# See: https://docs.pytorch.org/docs/stable/optim.html
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3              # students can tune
)

# 4. Early stopping
early_stopping = EarlyStopping(patience=5)

# 5. Training configuration
EPOCHS = 10  # maximum number of epochs

# Optional: store history for later analysis
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

# Training loop

for epoch in range(EPOCHS):

    # ----- Training -----
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = correct / total

    # ----- Validation -----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_accuracy = val_correct / val_total

    # Store metrics
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_accuracy)

    # Print epoch summary
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train loss: {train_loss:.4f} | Train acc: {train_accuracy:.4f}")
    print(f"  Val   loss: {val_loss:.4f} | Val   acc: {val_accuracy:.4f}")

    # ----- Early Stopping -----
    if early_stopping.step(val_loss):
        print("Early stopping triggered.")
        break

In [ ]:
# IMPORTANT: for fairness, run this only once after you finshined training and
# tuning your model using the test/val split

# we are interested in the test accuracy. report this number once you
# are finished to the professor
test_loss, test_acc = evaluate(model, test_loader, criterion)
print("Test accuracy:", test_acc)